# Function Calling and Tool Use

Function calling is the mechanism by which LLMs interact with external systems in a structured, type-safe way. Getting tool design right is one of the highest-leverage skills in agent engineering — poorly designed tools are the most common source of agent failures.

## Function Calling in the Anthropic API

Anthropic's tool use API lets you define tools as JSON schemas. The model decides when to call a tool, generates structured arguments, and receives the result as an observation. The protocol:
1. Define tools as a list of JSON Schema objects with name, description, and input_schema
2. Send the tool list with the user message
3. If the model decides to use a tool, it returns a `tool_use` content block
4. Execute the tool and return the result in a `tool_result` content block
5. Continue the conversation — the model may use more tools or produce a final answer

The key contract: tool schemas must precisely define what the tool does, what inputs it accepts, and what errors it can return. Ambiguous schemas lead to miscalled tools.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Callable
import json, re

@dataclass
class ToolSchema:
    name: str
    description: str
    input_schema: dict
    fn: Callable

    def to_api_dict(self) -> dict:
        return {
            'name': self.name,
            'description': self.description,
            'input_schema': self.input_schema,
        }

    def validate_input(self, args: dict) -> tuple:
        required = self.input_schema.get('required', [])
        properties = self.input_schema.get('properties', {})
        for field in required:
            if field not in args:
                return False, f'Missing required field: {field}'
        for k, v in args.items():
            if k not in properties:
                return False, f'Unknown field: {k}'
            expected_type = properties[k].get('type')
            if expected_type == 'number' and not isinstance(v, (int, float)):
                return False, f'Field {k} must be a number, got {type(v).__name__}'
            if expected_type == 'string' and not isinstance(v, str):
                return False, f'Field {k} must be a string'
        return True, None

    def call(self, args: dict) -> dict:
        valid, error = self.validate_input(args)
        if not valid:
            return {'error': error, 'success': False}
        try:
            result = self.fn(**args)
            return {'result': result, 'success': True}
        except Exception as e:
            return {'error': str(e), 'success': False}

# Well-designed tools
tools = [
    ToolSchema(
        name='get_weather',
        description='Get current weather for a city. Returns temperature in Celsius and conditions.',
        input_schema={
            'type': 'object',
            'properties': {
                'city': {'type': 'string', 'description': 'City name, e.g. London'},
                'units': {'type': 'string', 'enum': ['celsius', 'fahrenheit'], 'description': 'Temperature units'},
            },
            'required': ['city'],
        },
        fn=lambda city, units='celsius': {'temp': 18, 'conditions': 'Partly cloudy', 'city': city, 'units': units}
    ),
    ToolSchema(
        name='calculator',
        description='Evaluate a mathematical expression. Supports +, -, *, /, **, and parentheses.',
        input_schema={
            'type': 'object',
            'properties': {'expression': {'type': 'string', 'description': 'Math expression like "2 + 2" or "15 * 0.2"'}},
            'required': ['expression'],
        },
        fn=lambda expression: eval(expression, {'__builtins__': {}}, {})
    ),
]

# Test valid and invalid calls
test_calls = [
    ('get_weather', {'city': 'Paris', 'units': 'celsius'}),
    ('get_weather', {'city': 'Tokyo'}),  # missing units — OK, not required
    ('get_weather', {}),  # missing required city
    ('calculator', {'expression': '15 * 0.20'}),
    ('calculator', {'expression': '100 / 4', 'extra_field': 'bad'}),  # unknown field
]
tool_map = {t.name: t for t in tools}
for tool_name, args in test_calls:
    result = tool_map[tool_name].call(args)
    status = 'OK' if result['success'] else 'ERR'
    print(f'[{status}] {tool_name}({args}) -> {result}')

## Tool Design Principles

Good tool design follows these principles:

**Single responsibility**: each tool does one thing well. A tool that fetches weather AND converts currencies is harder for the model to use correctly than two separate tools.

**Descriptive names and descriptions**: the model uses the description to decide when to call the tool. 'search' is worse than 'search_web_for_current_information'. Include what the tool does NOT do.

**Explicit error returns**: return structured errors, not exceptions. The model needs to read and reason about errors to recover.

**Idempotency where possible**: read-only tools are always safe to retry. Write tools should include an idempotency key or check before executing.

**Input normalization**: accept flexible inputs, normalize internally. Accept 'New York', 'new york', 'NYC' — don't force the model to know the exact format.

In [ ]:
class MultiToolAgent:
    def __init__(self, tools: list, model_fn: Callable, max_steps: int = 8):
        self.tool_map = {t.name: t for t in tools}
        self.model = model_fn
        self.max_steps = max_steps

    def _tool_context(self) -> str:
        return '\n'.join(f'- {t.name}: {t.description}' for t in self.tool_map.values())

    def _parse_tool_call(self, response: str):
        m = re.search(r'CALL:(\w+)\s+(.+?)(?=CALL:|ANSWER:|$)', response, re.DOTALL)
        if m:
            name = m.group(1)
            try:
                args = json.loads(m.group(2).strip())
            except Exception:
                args = {'query': m.group(2).strip()[:100]}
            return name, args
        return None, None

    def run(self, query: str) -> str:
        history = [f'Task: {query}', f'Tools:\n{self._tool_context()}']
        for step in range(self.max_steps):
            response = self.model('\n'.join(history))
            history.append(f'Agent: {response[:100]}')
            if 'ANSWER:' in response:
                return response.split('ANSWER:')[-1].strip()
            name, args = self._parse_tool_call(response)
            if name and name in self.tool_map:
                result = self.tool_map[name].call(args)
                obs = f'ToolResult[{name}]: {json.dumps(result)[: 150]}'
                history.append(obs)
        return 'Could not complete task within step limit'

step_c = [0]
def demo_model(context):
    step_c[0] += 1
    if step_c[0] == 1:
        return 'I need the weather in Paris. CALL:get_weather {"city": "Paris"}'
    if step_c[0] == 2:
        return 'ANSWER: It is currently 18°C and partly cloudy in Paris.'
    return 'ANSWER: Done.'

agent = MultiToolAgent(tools, demo_model)
result = agent.run('What is the weather like in Paris right now?')
print(f'Result: {result}')

## Error Handling and Retry Logic

Production tool-use agents need robust error handling:

**Tool errors**: when a tool returns an error, the agent should read the error message, diagnose the issue (wrong arguments? network error? rate limit?), and either fix the call or try an alternative approach.

**Retry with backoff**: for transient errors (rate limits, timeouts), retry with exponential backoff. For semantic errors (wrong query format), revise the call rather than retrying.

**Graceful degradation**: if a tool is unavailable, the agent should fall back to its parametric knowledge with explicit uncertainty: 'I cannot reach the weather API, but based on historical data for Paris in April...'